# 🔬 Interactive Molecular Dynamics Streaming Workshop

## Real-Time streaming with IMDv3 and MDAnalysis

Welcome! In this hands-on workshop, you'll learn to stream live molecular simulation data using the IMDv3 streaming implementation and analyze such data in real-time with MDAnalysis.

### 🎯 What You'll Learn

- Connect to live MD simulations via IMD streaming and MDAnalysis
- Perform real-time monitoring of simulations
- Analyze and visualize MD properties on-the-fly
- Implement adaptive and selective sampling strategies

### 🧬 System: Lysozyme in Water

- **Protein**: Lysozyme, 129 residues
- **Solvent**: 9,467 TIP3P water molecules
- **Ions**: 27 Na⁺, 35 Cl⁻
- **Total atoms**: 30,423
- **Box size**: 6.9 × 6.9 × 6.9 nm³
- **MD Engine**: GROMACS, LAMMPS and NAMD with IMDv3

### 📅 Workshop Schedule (60 minutes)

| Time | Section | Topic |
|------|---------|-------|
| 0-5 min | **Setup** | Connect to simulation |
| 5-15 min | **Application 1** | Continuous monitoring |
| 15-25 min | **Application 2** | Monitoring with breaks |
| 25-35 min | **Application 3** | Adaptive and selective sampling |
| 35-45 min | **Application 4** | Live visualization |
| 45-52 min | **Exercise 1** | Hydration shell analysis |
| 52-60 min | **Exercise 2** | Radius of gyration |

Let's get started! 🚀

---

## Getting Started

### Simulation Engine - GROMACS

For the purposes of this workshop, we will start by running **GROMACS** with IMDv3 streaming. 
(LAMMPS and NAMD input and run scripts have also been provided for your reference in `sample_simulation/LAMMPS/` and `sample_simulation/NAMD/`.)

### Required Input Files

Our GROMACS simulation needs:
- `start.gro` - Initial structure (lysozyme + water + ions)
- `topol.top` - Topology file (atom types, bonds, interactions)
- `index.ndx` - Atom groups (contains custom **SOLU**=protein, **SOLV**=water+ions)
- `input-streaming.mdp` - MD parameters with IMDv3 settings

### Key MDP Parameters for IMDv3 Streaming

```bash
# IMDv3 Streaming Configuration
IMD-version = 3                # Use IMDv3 protocol
IMD-nst     = 10               # Stream every 10 steps (20 fs)
IMD-coords  = Yes              # Send coordinates
IMD-group   = System           # Stream all atoms
```

**Why we need `index.ndx`:** The MDP file uses custom CHARMM-GUI groups (`SOLU`/`SOLV`) that don't exist in default GROMACS groups. Without this file, grompp will fail!

### Step 1: Start the MD Simulation

Open a **new terminal** and run:

```bash
cd /workspaces/imd-workshop-2025/workshop/sample_simulation/GROMACS
./run.sh
```

You should see:
```
IMD: Will wait until I have a connection and IMD_GO orders.
```

### Step 2: Connect to the Simulation from MDAnalysis

Run the cell below to establish connection:


In [ ]:
import MDAnalysis as mda

# Connect to live simulation
u = mda.Universe("sample_simulation/GROMACS/input/start.gro", "imd://localhost:8889", buffer_size=100*1024**2)

print("✅ Connected to simulation!")
print(f"📦 System: {u.atoms.n_atoms} atoms")
print(f"🧬 Protein: {u.select_atoms('protein').n_residues} residues")
print(f"💧 Waters: {u.select_atoms('resname TIP3').n_residues} molecules")


### Step 3: Accessing Simulation Data and Performing Analysis

One can now access simulation data in real-time using MDAnalysis, and looping over trajectory frames.

All analysis cells in this workshop use the **try-except-finally** pattern for safe execution:

```python
try:
    for ts in u.trajectory:
        # Your analysis code here
        pass
except KeyboardInterrupt:
    print("Analysis stopped by user")
finally:
    u.trajectory.close()  # Always cleanup connection
```

**To stop analysis:** Click the **Stop button (⏹️)** in the cell toolbar or press **Interrupt Kernel**. 

The `finally` block ensures the IMD connection is properly closed, even if you interrupt the cell.

### Step 4: Stopping and Restarting the Simulation

**To stop the simulation:**
- In the terminal running `./run.sh`, press **Ctrl+C**
- GROMACS will shut down gracefully

**To restart:**
```bash
cd /workspaces/imd-workshop-2025/workshop/sample_simulation/GROMACS
./run.sh
```

Then **reconnect from MDAnalysis** by rerunning the connection cell.


---

## 📊 Application 1: Continuous Monitoring

Monitor simulation using streamed data as it arrives in real-time.

### Why is this useful?
- Track molecular properties continuously (positions, velocities, forces, box dimensions)
- Perform simple calculations on-the-fly (distances, angles, etc.)
- Identify anomalies early

### Example 1: Basic Monitoring
Print position, velocity, force, and box size for the first atom at each timestep.


In [ ]:
import MDAnalysis as mda

u = mda.Universe("sample_simulation/GROMACS/input/start.gro", "imd://localhost:8889", buffer_size=100*1024**2)

# Get the first atom from the topology
atom = u.atoms[0]

print("Starting basic monitoring...")
print("Press the Stop button (⏹️) to interrupt\n")
print(" step time [           position          ] [           velocity          ] [             force           ] [             box             ]")

try:
    for ts in u.trajectory:
        print(f'{ts.data["step"]} {ts.time:8.3f} {atom.position} {atom.velocity} {atom.force} {u.dimensions[0:3]}')

except KeyboardInterrupt:
    print("\n\nMonitoring stopped by user")
except Exception as e:
    print(f"\n\nError: {e}")
finally:
    u.trajectory.close()
    print(f"\nAnalyzed {ts.frame + 1} frames")


### Example 2: Simple Analysis On-The-Fly
Monitor the distance between N-terminus and C-terminus backbone carbons in real-time.


In [ ]:
import MDAnalysis as mda
import numpy as np

u = mda.Universe("sample_simulation/GROMACS/input/start.gro", "imd://localhost:8889", buffer_size=100*1024**2)

# Select N-terminus (residue 2) and C-terminus (residue 129) alpha carbons
nter = u.select_atoms("protein and resid 2 and name CA")[0]
cter = u.select_atoms("protein and resid 129 and name CA")[0]

try:
    for ts in u.trajectory:
        distance = np.linalg.norm(nter.position - cter.position)
        print(f"Frame {ts.frame:4d} | Time: {ts.time:8.2f} ps | N-C Distance: {distance:.2f} Å")

except Exception as e:
    print(f"\nError: {e}")
finally:
    u.trajectory.close()
    print(f"\nAnalyzed {ts.frame + 1} frames")

---

## ⏸️ Application 2: Monitoring with Breaks

Connect to simulation intermittently - analyze frames, disconnect, pause, then reconnect to continue.

### Why is this useful?
- Useful for long simulations where continuous monitoring isn't needed
- Allows time for post-processing between batches
- Use `continue_after_disconnect=True` to keep simulation running during disconnections

**Example**: Monitor N-C terminal distance for 20 frames, pause 5 seconds, reconnect, repeat.


In [ ]:
import MDAnalysis as mda
import numpy as np
import time

frames_per_batch = 20
pause_duration = 10  # seconds

# ===== BATCH 1 =====
print("="*60)
print("BATCH 1")
print("="*60)

u = mda.Universe("sample_simulation/GROMACS/input/start.gro", "imd://localhost:8889", buffer_size=100*1024**2, continue_after_disconnect=True)

nter = u.select_atoms("protein and resid 2 and name CA")[0]
cter = u.select_atoms("protein and resid 129 and name CA")[0]

distances_1 = []
frame_count = 0

try:
    for ts in u.trajectory:
        distance = np.linalg.norm(nter.position - cter.position)
        distances_1.append(distance)
        frame_count += 1
        
        print(f"  Step {ts.data['step']} | Time: {ts.time:8.2f} ps | Distance: {distance:.2f} Å")
        
        if frame_count >= frames_per_batch:
            break

except Exception as e:
    print(f"Error: {e}")
finally:
    u.trajectory.close()

print(f"\n✅ Batch 1 complete: {frame_count} frames")
print(f"   Distance range: {min(distances_1):.2f} - {max(distances_1):.2f} Å")
print(f"   Average: {np.mean(distances_1):.2f} Å")

print(f"\n⏸️  Pausing for {pause_duration} seconds...")
print("   (Simulation continues running during pause)\n")
time.sleep(pause_duration)

# ===== BATCH 2 =====
print("="*60)
print("BATCH 2")
print("="*60)

u = mda.Universe("sample_simulation/GROMACS/input/start.gro", "imd://localhost:8889", buffer_size=100*1024**2, continue_after_disconnect=True)

nter = u.select_atoms("protein and resid 2 and name CA")[0]
cter = u.select_atoms("protein and resid 129 and name CA")[0]

distances_2 = []
frame_count = 0

try:
    for ts in u.trajectory:
        distance = np.linalg.norm(nter.position - cter.position)
        distances_2.append(distance)
        frame_count += 1

        print(f"  Step {ts.data['step']} | Time: {ts.time:8.2f} ps | Distance: {distance:.2f} Å")

        if frame_count >= frames_per_batch:
            break

except Exception as e:
    print(f"Error: {e}")
finally:
    u.trajectory.close()

print(f"\n✅ Batch 2 complete: {frame_count} frames")
print(f"   Distance range: {min(distances_2):.2f} - {max(distances_2):.2f} Å")
print(f"   Average: {np.mean(distances_2):.2f} Å")


---

## 🎯 Application 3: Adaptive and Selective Sampling

Save frames at **logarithmic intervals** and write **only specific parts** of the system to disk.

### Why is this useful?
- Dramatically reduce trajectory file sizes (save only protein, not water)
- Capture early dynamics densely, later dynamics sparsely (1, 2, 4, 8, 16, 32...)
- Focus storage on molecules of interest

**Example**: Save protein-only frames at powers of 2.


In [ ]:
import MDAnalysis as mda
import numpy as np

u = mda.Universe("sample_simulation/GROMACS/input/start.gro", "imd://localhost:8889", buffer_size=100*1024**2)

# Select only the protein (exclude water and ions)
protein = u.select_atoms("protein")

saved_frames = []
frame_count = 0

print(f"Starting adaptive sampling: Protein only ({protein.n_atoms} atoms)")
print(f"Saving at powers of 2: frames 1, 2, 4, 8, 16...\n")

try:
    with mda.Writer("sample_simulation/GROMACS/output/protein_log2.trr", protein.n_atoms) as writer:
        for ts in u.trajectory:
            frame_count += 1
            
            # Save at powers of 2: frames 1, 2, 4, 8, 16, 32, 64, 128...
            log_val = np.log2(frame_count)
            if log_val == int(log_val):
                writer.write(protein)
                saved_frames.append(frame_count)
                print(f"✅ Saved frame {frame_count:4d} | Time: {ts.time:8.2f} ps")

except Exception as e:
    print(f"\nError: {e}")
finally:
    u.trajectory.close()

print(f"\nTotal: {frame_count} frames | Saved: {len(saved_frames)} frames ({saved_frames})")


---

## 📈 Application 4: Live Visualization

Create **real-time plots** that update as the simulation runs.

### Why is this useful?
- Immediate visual feedback on simulation behavior
- Monitor protein folding/unfolding in real-time
- Detect problems before wasting compute time
- Great for presentations and demonstrations

**Example**: Live plot of protein radius of gyration (Rg) - a measure of protein compactness.


In [ ]:
import MDAnalysis as mda
from graph_utils import live_plot

u = mda.Universe("sample_simulation/GROMACS/input/start.gro", "imd://localhost:8889", buffer_size=100*1024**2)

# Select protein
protein = u.select_atoms("protein")

# Create live plot
plot = live_plot(title="Protein Radius of Gyration", ylabel="Rg (Å)", update_interval=1)

try:
    for ts in u.trajectory:
        # Calculate radius of gyration using MDAnalysis method
        rg = protein.radius_of_gyration()
        plot['update'](ts.time, rg)

except Exception as e:
    print(f"\nError: {e}")
finally:
    u.trajectory.close()
    plot['close']()
    print("Live visualization complete!")


---

## 🎯 Exercise 1: Charge Density Analysis

**Your Task**: Calculate the **total charge within 4 Å of the protein** at each timestep.

This is an important property for understanding:
- Electrostatic environment around the protein
- Ion distribution near protein surface
- Screening effects in solution

### 📚 Helpful Tips

**Selecting all atoms (water + ions):**
```python
solvent = u.select_atoms('not protein')
```

**Distance-based selection** (solvent within 4 Å of protein):
```python
# Use updating=True to recalculate selection each frame!
nearby = u.select_atoms('not protein and around 4 protein', updating=True)
```

**Getting atom charges:**
```python
charges = selection.charges  # Array of partial charges for each atom
```

**Calculating total charge:**
```python
total_charge = np.sum(charges)  # Sum all charges in selection
```

**Bonus Challenge**: Create a live visualization of charge density over time! 📊

💡 **Hint**: Use `from graph_utils import live_plot` for easy live plotting!

🔍 See `solutions.ipynb` if you need help!

In [ ]:
from imdclient.IMD import IMDReader
import MDAnalysis as mda
import numpy as np
from graph_utils import live_plot

u = mda.Universe(
    "sample_simulation/GROMACS/input/start.gro",
    "imd://localhost:8889",
    buffer_size=100*1024**2
)

try:
    ## Your code here!
    ## 
    ## Steps:
    ## 1. Create selection for nearby solvent: nearby = u.select_atoms('not protein and around 4 protein', updating=True)
    ## 2. Create live plot: plot = live_plot("Charge Density within 4 Å", ylabel="Total Charge (e)")
    ## 3. Loop through frames
    ## 4. Calculate total_charge = np.sum(nearby.charges)
    ## 5. Update plot: plot['update'](ts.time, total_charge)
    
    pass  # Remove this and add your code

except Exception as e:
    print(f"Error: {e}")
finally:
    u.trajectory.close()

---

## 🎯 Exercise 2: Backbone RMSD

**Your Task**: Calculate the **Root Mean Square Deviation (RMSD)** of the protein backbone relative to the starting structure.

RMSD measures how much the protein structure deviates from its initial conformation - a fundamental analysis in MD simulations!

### 📐 What is RMSD?

RMSD quantifies structural similarity between two conformations:

$$\text{RMSD} = \sqrt{ \frac{1}{N} \sum_{i=1}^{N} (r_i - r_i^{\text{ref}})^2 }$$

where $r_i$ are atom positions and $r_i^{\text{ref}}$ are reference positions.

### 📚 Helpful Tips

**Select protein backbone atoms:**
```python
backbone = u.select_atoms("protein and backbone")
```

**Save reference positions from first frame:**
```python
# Inside the loop, on first frame:
if ts.frame == 0:
    reference_positions = backbone.positions.copy()
```

**Calculate RMSD using MDAnalysis:**
```python
from MDAnalysis.analysis import rms

# Calculate RMSD between current and reference
rmsd_value = rms.rmsd(backbone.positions, reference_positions, superposition=True)
```

**What `superposition=True` does:** Aligns structures before calculating RMSD (removes rotation/translation)

**Bonus Challenge**: 
- Create a live visualization of RMSD over time! 📊
- Calculate RMSD for different regions (e.g., active site vs full protein)

🔍 See `solutions.ipynb` if you need help!

📖 More info: [MDAnalysis RMSD Documentation](https://docs.mdanalysis.org/stable/documentation_pages/analysis/rms.html)

In [ ]:
from imdclient.IMD import IMDReader
import MDAnalysis as mda
from MDAnalysis.analysis import rms
import numpy as np
from graph_utils import live_plot

u = mda.Universe(
    "sample_simulation/GROMACS/input/start.gro",
    "imd://localhost:8889",
    buffer_size=100*1024**2
)

try:
    ## Your code here!
    ## 
    ## Steps:
    ## 1. Select backbone atoms: backbone = u.select_atoms("protein and backbone")
    ## 2. Create variable to store reference: reference_positions = None
    ## 3. Create live plot: plot = live_plot("Backbone RMSD", ylabel="RMSD (Å)")
    ## 4. Loop through frames
    ## 5. On first frame (ts.frame == 0): save reference_positions = backbone.positions.copy()
    ## 6. Calculate RMSD: rmsd_value = rms.rmsd(backbone.positions, reference_positions, superposition=True)
    ## 7. Update plot: plot['update'](ts.time, rmsd_value)
    
    pass  # Remove this and add your code

except Exception as e:
    print(f"Error: {e}")
finally:
    u.trajectory.close()

---

## 🎉 Congratulations!

You've completed the IMD streaming workshop! You now know how to:

✅ Connect to live MD simulations via IMDv3  
✅ Perform continuous real-time analysis  
✅ Implement batched monitoring strategies  
✅ Create adaptive sampling workflows  
✅ Build live visualizations  
✅ Calculate custom properties (hydration, Rg, distances)  

### 🚀 Next Steps

- **Try other MD engines**: LAMMPS and NAMD example scripts in `sample_simulation/LAMMPS/` and `sample_simulation/NAMD/`
- **Explore more analyses**: RMSF, hydrogen bonds, secondary structure, radial distribution function et al.
- **Implement your own**: Use these patterns for your research!

### 📚 Resources

- [MDAnalysis Documentation](https://docs.mdanalysis.org/)
- [IMDClient GitHub](https://github.com/Becksteinlab/imdclient)
- [Workshop Materials](https://github.com/amruthesht/imd-workshop-2025)

**Questions?** Check out the [MDAnalysis Discord](https://discord.gg/fXHy8z) or [GitHub Discussions](https://github.com/MDAnalysis/mdanalysis/discussions)!

Happy analyzing! 🔬✨